In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/임베디드/Mobius")  # mobius_client.py가 있는 폴더 경로

import base64
import io
...
from mobius_client import get_latest, post_value

Cell 1:설치

- transformers — Qwen3-VL 모델을 불러오고 실행하는 데 필요한 HuggingFace 라이브러리
- accelerate — GPU에 모델을 효율적으로 올리는 데 필요
fastapi, uvicorn — 나중에 Cell 4에서 HTTP 서버 만들 때 필요
- pyngrok — Colab을 외부에서 접근 가능하게 터널 뚫는 용도
- nest_asyncio — Colab 노트북 환경에서 서버를 실행하기 위한 호환성 도구
- pillow — 이미지 처리
- qwen-vl-utils — Qwen-VL 계열 모델 전용 유틸리티

In [ ]:
!pip install -q transformers accelerate fastapi uvicorn pyngrok nest_asyncio pillow qwen-vl-utils
!pip install -q openai-whisper
!apt-get -qq install -y ffmpeg

Cell2:모델 로드

Qwen3-VL-32B-Instruct 모델 가중치를 HuggingFace에서 다운로드해서 GPU메모리에 올리는 단계

In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype="auto",         
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("모델 로드 완료")

Cell 3 : 판단 함수

In [ ]:
import base64
import io
import json
import re
import tempfile
import os
from PIL import Image
from mobius_client import get_latest, post_value

SYSTEM_PROMPT = """\
너는 가족 감정 케어 시스템의 판단 엔진이다.
식탁 같은 공용 공간에서, 두 명 이상이 자리에 앉았을 때 대화와 사진을 보고
지금 분위기를 파악한 뒤, 화제를 추천할지 말지 결정한다.

[입력 정보]
- images: 좌석별 사진 (좌석당 1장, 각 사진에 seat_id가 붙어있음). 표정/자세 등 시각적 단서.
- transcript: 최근 대화를 STT로 변환한 텍스트. 비어 있으면 사진(들)만으로 판단한다.
- 추가 정보: loudness_db(목소리 크기), temp_c/humidity_pct(온습도) 등

[판단 방식]
1. 먼저 각 seat_id별로 표정/자세에서 드러나는 분위기를 개별적으로 파악한다.
2. 그다음 두 좌석의 분위기를 종합해서 그 공간 전체의 분위기와 상호작용 톤
   (서로 잘 호응하는지, 한쪽이 침묵/회피하는지, 둘 다 가라앉아 있는지 등)을 판단한다.
3. 온습도는 보조 신호로만 사용한다. 표정/대화에서 감정이 명확히 드러나면 그것을 우선하고,
   temp_c/humidity_pct는 표정·대화만으로는 애매하게 갈리는 경우에만 판단을 보정하는 용도로 쓴다.
   - 대략 temp_c 28 이상 & humidity_pct 65 이상처럼 무덥고 습한 상태가 지속되면,
     calm과 tired/angry 경계에 있는 애매한 경우 tired 또는 angry 쪽으로 살짝 기울여 판단할 수 있다.
   - 표정/대화가 뚜렷하게 calm이나 joy인 경우에는 온습도만으로 mood를 바꾸지 않는다.

[감정 스펙트럼 (detected_mood)]
- joy (기쁨/신남): 즐겁거나 신나 보임
- calm (평온): 특별한 변화 없이 안정적인 상태
- tired (피곤/무기력): 지치거나 기운 없어 보임
- sad (슬픔/속상함): 가라앉아 있거나 속상해 보임
- angry (화남/짜증): 짜증나거나 화가 난 듯 보임

[반응 규칙]
- action은 기본적으로 "gathering_topic"이다. 대화가 자연스럽게 잘 흐르고 있어서
  개입이 전혀 필요 없다고 판단되면 "no_action"을 쓸 수 있다.
- mood에 따라 화제 방향을 다음과 같이 구분한다:
  - joy → 지금의 좋은 기분을 더 이야기하고 확장할 수 있는 주제를 추천한다.
    이미 나온 좋은 일이 있다면 그걸 더 물어보게 유도하고, 없다면 "요즘 있었던 좋은 일/기대되는 일"처럼
    스스로 좋은 경험을 꺼내 말하게 만드는 주제를 추천한다.
  - calm → 평소 대화 소재가 될 만한 가벼운 일상 주제 (음식, 요즘 본 것, 근황 등). 특별한 개입 의도 없이 자연스럽게.
  - tired → 생각을 많이 안 해도 답할 수 있는, 가볍게 떠올리기만 하면 되는 주제
    (좋아하는 음식, 최근 편하게 즐긴 것 등 가벼운 긍정 기억을 스치듯 꺼내는 정도). 길게 이어가길 요구하지 않는다.
  - sad → 무슨 일이 있었는지 캐묻거나 감정을 직접 다루지 않는다. 대신 가볍고 안전한, 부담 없이 스쳐 지나갈 수 있는
    긍정적인 소재(좋아하는 음식, 편하게 봤던 것 등)를 아주 담백하게 던진다. 깊이 파고들지 않는다.
  - angry → 화제를 추천하기보다, 잠깐 멈추고 마음을 가라앉힐 수 있게 부드럽게 권유한다.
    "무슨 일이야?" 같은 원인을 캐묻는 말이나, 지금 상황을 지적하는 말은 하지 않는다.
    대신 "한 템포 쉬고 얘기해볼까요?", "따뜻한 차 한 잔 마시면서 잠깐 쉬어가는 건 어떨까요?" 처럼
    부담 없이 잠깐 멈추고 스스로를 가다듬을 수 있게 짧고 다정하게 제안한다.
    상황이나 원인을 연상시키는 표현은 피한다.
- 어떤 mood든 지금 상황을 직접 언급하거나 지적하지 않는다 (예: "화가 나 보이네요", "오늘 피곤해 보이네요" 같은 표현 금지).
- 어떤 mood든 돈, 성적, 갈등 소재는 절대 다시 언급하거나 주제로 제안하지 않는다.
- 단정적 진단 톤 금지.
- transcript가 비어 있으면 사진(들)만으로 판단한다.
- lcd 문구는 한글 기준 최대 20자.
- 반드시 JSON 객체 하나만 출력한다. 마크다운 코드블록(```) 금지.
- detected_mood 필드에 위 5가지 중 하나를 반드시 포함한다 (no_action이어도 포함).

[출력 형식 예시]
{"action": "gathering_topic", "detected_mood": "calm", "lcd": "요즘 재밌게 본 영화 있어요?"}
또는
{"action": "no_action", "detected_mood": "calm"}
"""

_whisper_model = None


def _load_whisper():
    global _whisper_model
    if _whisper_model is None:
        import whisper
        print("[decide] Whisper 모델 로드 중...")
        _whisper_model = whisper.load_model("small")
        print("[decide] Whisper 모델 로드 완료")
    return _whisper_model


def _transcribe_audio_base64(audio_b64: str) -> str:
    try:
        model = _load_whisper()
        audio_bytes = base64.b64decode(audio_b64)
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp.write(audio_bytes)
            tmp_path = tmp.name
        try:
            result = model.transcribe(tmp_path, language="ko", fp16=True)
            return result["text"].strip()
        finally:
            os.remove(tmp_path)
    except Exception as e:
        print(f"[decide] STT 실패: {e}")
        return ""


def _fetch_env_from_mobius() -> dict:
    """Mobius의 tem/hum 컨테이너에서 최신 온습도 값을 가져옴. 실패하면 빈 dict."""
    env = {}
    temp = get_latest("tem")
    hum = get_latest("hum")
    if temp is not None:
        env["temp_c"] = temp
    if hum is not None:
        env["humidity_pct"] = hum
    return env

_prev_ac_state = "off"  


def _decide_ac(temp_c) -> str:
    global _prev_ac_state
    try:
        t = float(temp_c)
    except (TypeError, ValueError):
        return _prev_ac_state

    if t >= 25:
        _prev_ac_state = "on"
    elif t <= 20:
        _prev_ac_state = "off"

    return _prev_ac_state


def decide(payload: dict) -> dict:
    
    transcript = ""
    if payload.get("audio_base64"):
        transcript = _transcribe_audio_base64(payload["audio_base64"])
        print(f"[decide] STT 결과: {transcript}")

    
    images = []
    seat_labels = []
    for item in payload.get("images", []) or []:
        b64 = item.get("image_base64")
        if not b64:
            continue
        img = Image.open(io.BytesIO(base64.b64decode(b64)))
        images.append(img)
        seat_labels.append(item.get("seat_id", "unknown_seat"))

  
    env = _fetch_env_from_mobius()
    print(f"[decide] Mobius 온습도: {env}")

    extra = {
        k: v
        for k, v in payload.items()
        if k not in ("images", "audio_base64")
    }
    extra.update(env)

    user_text = (
        f"대화 텍스트(STT): {transcript or '(없음)'}\n"
        f"사진 좌석 순서: {seat_labels if seat_labels else '(사진 없음)'}\n"
        f"추가 정보: {json.dumps(extra, ensure_ascii=False)}\n\n"
        f"위 정보를 보고 JSON으로만 답해."
    )

    content = []
    for img in images:
        content.append({"type": "image", "image": img})
    content.append({"type": "text", "text": user_text})

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": content},
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=[text],
        images=images if images else None,
        return_tensors="pt",
    ).to(model.device)

    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    output_text = processor.batch_decode(
        output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]

    match = re.search(r"\{.*\}", output_text, re.DOTALL)
    if not match:
        print(f"[decide] JSON 못 찾음, 원문: {output_text[:200]}")
        result = {"action": "no_action"}
    else:
        try:
            result = json.loads(match.group())
        except json.JSONDecodeError:
            print(f"[decide] JSON 파싱 실패, 원문: {output_text[:200]}")
            result = {"action": "no_action"}

    if result.get("lcd") and len(result["lcd"]) > 20:
        result["lcd"] = result["lcd"][:20]

    VALID_MOODS = {"joy", "calm", "tired", "sad", "angry"}
    if result.get("detected_mood") not in VALID_MOODS:
        result["detected_mood"] = "calm"

    VALID_ACTIONS = {"gathering_topic", "no_action"}
    if result.get("action") not in VALID_ACTIONS:
        result["action"] = "no_action"

    result["ac"] = _decide_ac(env.get("temp_c"))

    status = post_value("cmd", json.dumps(result, ensure_ascii=False))
    print(f"[decide] Mobius cmd 컨테이너 POST 상태: {status}")

    return result

Cell 4: FastAPI 서버 + ngrok 터널

In [ ]:
from fastapi import FastAPI, Request
import uvicorn
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()
app = FastAPI()


@app.post("/decide")
async def decide_endpoint(request: Request):
    payload = await request.json()
    try:
        return decide(payload)
    except Exception as e:
        print(f"[server] 처리 중 오류: {e}")
        return {"action": "no_action"}


NGROK_AUTH_TOKEN = "<YOUR_NGROK_AUTH_TOKEN>" 
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
print("=" * 60)
print(f"RPi4 쪽 COLAB_DECIDE_URL 환경변수에 이 값을 넣으세요:")
print(f"{public_url}/decide")
print("=" * 60)

config = uvicorn.Config(app, port=8000)
server = uvicorn.Server(config)
await server.serve()